In [2]:
import pandas as pd
import numpy as np

RAW_PATH = "data/raw/lending_club.csv"  

df = pd.read_csv(RAW_PATH, low_memory=False)
print(f"Shape: {df.shape}")
df.head(3)

Shape: (2260701, 151)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
print(df.dtypes.value_counts())
print()
print(df.dtypes.sort_index())

float64    113
object      38
Name: count, dtype: int64

acc_now_delinq               float64
acc_open_past_24mths         float64
addr_state                    object
all_util                     float64
annual_inc                   float64
                              ...   
total_rev_hi_lim             float64
url                           object
verification_status           object
verification_status_joint     object
zip_code                      object
Length: 151, dtype: object


In [4]:
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_pct.head(30)

member_id                                     100.000000
orig_projected_additional_accrued_interest     99.617331
hardship_reason                                99.517097
hardship_payoff_balance_amount                 99.517097
hardship_last_payment_amount                   99.517097
payment_plan_start_date                        99.517097
hardship_type                                  99.517097
hardship_status                                99.517097
hardship_start_date                            99.517097
deferral_term                                  99.517097
hardship_amount                                99.517097
hardship_dpd                                   99.517097
hardship_loan_status                           99.517097
hardship_length                                99.517097
hardship_end_date                              99.517097
settlement_status                              98.485160
debt_settlement_flag_date                      98.485160
settlement_term                

In [ ]:
near_empty_cols = missing_pct[missing_pct >= 95].index.tolist()
print(f"{len(near_empty_cols)} columns >= 95% missing:")
near_empty_cols

34 columns >= 95% missing:


['member_id',
 'orig_projected_additional_accrued_interest',
 'hardship_reason',
 'hardship_payoff_balance_amount',
 'hardship_last_payment_amount',
 'payment_plan_start_date',
 'hardship_type',
 'hardship_status',
 'hardship_start_date',
 'deferral_term',
 'hardship_amount',
 'hardship_dpd',
 'hardship_loan_status',
 'hardship_length',
 'hardship_end_date',
 'settlement_status',
 'debt_settlement_flag_date',
 'settlement_term',
 'settlement_percentage',
 'settlement_date',
 'settlement_amount',
 'sec_app_mths_since_last_major_derog',
 'sec_app_revol_util',
 'revol_bal_joint',
 'sec_app_inq_last_6mths',
 'sec_app_num_rev_accts',
 'sec_app_open_acc',
 'sec_app_earliest_cr_line',
 'sec_app_fico_range_high',
 'sec_app_mort_acc',
 'sec_app_open_act_il',
 'sec_app_fico_range_low',
 'sec_app_collections_12_mths_ex_med',
 'sec_app_chargeoff_within_12_mths']

In [6]:
print(df['loan_status'].value_counts())

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64


In [ ]:
RESOLVED_STATUSES = {
    "Fully Paid": 0,
    "Charged Off": 1,
    "Default": 1
}

resolved = df[df['loan_status'].isin(RESOLVED_STATUSES.keys())].copy()
resolved['target'] = resolved['loan_status'].map(RESOLVED_STATUSES)

n_before, n_after = len(df), len(resolved)
print(f"Rows before: {n_before:,} | after filtering to resolved loans: {n_after:,} " f"({n_after/n_before:.1%} retained)")
print()
print("Default rate:", resolved['target'].mean().round(4))

Rows before: 2,260,701 | after filtering to resolved loans: 1,345,350 (59.5% retained)

Default rate: 0.1996


In [ ]:
LEAKAGE_COLS = [
    'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
    'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
    'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d',
    'out_prncp', 'out_prncp_inv',
    'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low',
    'loan_status', 'policy_code'
]

pattern_leakage = [c for c in resolved.columns
                    if c.startswith('hardship_') or c.startswith('settlement_')
                    or c.startswith('debt_settlement_flag')]

ID_TEXT_COLS = ['id', 'member_id', 'url', 'desc', 'title', 'emp_title']

drop_cols = set(LEAKAGE_COLS) | set(pattern_leakage) | set(ID_TEXT_COLS) | set(near_empty_cols)
drop_cols &= set(resolved.columns)  

print(f"Dropping {len(drop_cols)} columns")
cleaned = resolved.drop(columns=list(drop_cols))
print(f"Remaining shape: {cleaned.shape}")

Dropping 58 columns
Remaining shape: (1345350, 94)


In [11]:
cleaned.to_csv("data/processed/loans_stage1.csv", index=False)
cleaned.head(3)

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,...,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,disbursement_method,target
0,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,10+ years,MORTGAGE,...,76.9,0.0,0.0,0.0,178050.0,7746.0,2400.0,13734.0,Cash,0
1,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,10+ years,MORTGAGE,...,97.4,7.7,0.0,0.0,314017.0,39475.0,79300.0,24667.0,Cash,0
2,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,10+ years,MORTGAGE,...,100.0,50.0,0.0,0.0,218418.0,18696.0,6200.0,14877.0,Cash,0
